# Session 1: Basics: how LLM inference works, and what limits it

*LLM Inference Engineering*, session 1 of 3. Build the generation loop by hand, work out where GPU memory goes, measure your GPU, then do the same things with vLLM and see the numbers line up.

## How the series is divided

| Session | The question it answers | Topics |
|---|---|---|
| **1. Basics** *(this one)* | What happens between a prompt and its answer, and why is it slow? | Tokens, prefill and decode, and the KV cache, built from scratch; Memory math: weights, KV cache, what fits on a GPU; The GPU underneath: memory hierarchy, roofline, the decode speed limit; The metrics: TTFT, TPOT, throughput, goodput, cost per token; First steps with vLLM: offline engine, sampling, the OpenAI server, a load test |
| **2. Intermediate** | Which tricks make serving fast, and what does each one actually do? | Continuous batching and PagedAttention: a scheduler and a block allocator in code; Prefix caching and chunked prefill; Kernels: FlashAttention, CUDA graphs, the roofline measured; Quantization: INT8, FP8, INT4, NF4, AWQ, GPTQ, MXFP4, built and compared; Speculative decoding: draft-and-verify from scratch, n-gram, EAGLE-3, MTP; Structured outputs, scoring, embeddings, LoRA; vLLM vs SGLang |
| **3. Advanced** | How do you serve a model bigger than one GPU, reliably, for many users? | How GPUs connect: NVLink, PCIe, InfiniBand, NCCL, measured; Tensor, pipeline, data and expert parallelism in code, and when to use which; Big models: 70B to DeepSeek and Kimi on 8 GPUs, multi-node; Disaggregated prefill and decode, KV offload, TensorRT-LLM and Dynamo; Hosting open-weight models: recipes, tool calling, reasoning, vision, local machines; Production: Docker and Kubernetes, metrics, priorities, the tuning playbook |

| | |
|---|---|
| **Needs** | A laptop for the from-scratch and maths sections; one NVIDIA GPU (16 GB+) with vLLM for the rest. |
| **Labels** | *(no GPU needed)* runs on a laptop; *(reading)* has nothing to execute; everything else needs an NVIDIA GPU |
| **Run order** | Top to bottom. Engines are started and stopped per section, so you can also jump to one after running the setup cell. |

In [ ]:
%pip install -q vllm openai matplotlib pandas transformers huggingface_hub pyarrow

In [ ]:
# Shared helpers live in infer_utils.py next to this notebook:
# MODEL, MAX_LEN, PORT, BASE, launch, vllm_serve, stop, shutdown, measure_stream, bench_openai, gpu_mem
from infer_utils import *
try:
    from vllm import LLM, SamplingParams
except ImportError:
    print("vLLM is not installed here; the from-scratch and reading sections still run.")
import os, gc, time, json, subprocess
import numpy as np, pandas as pd, requests
from openai import OpenAI, AsyncOpenAI
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL)          # the default model's tokenizer, used for token counts
client = OpenAI(base_url=f"{BASE}/v1", api_key="not-needed")
results = {}                                         # benchmark tables, keyed by engine
print(env_report())

## Contents

1. Setup & GPU check
2. Tokens: what the model actually sees *(no GPU needed)*
3. Generation from scratch: prefill, decode and the KV cache in 30 lines *(runs on CPU)*
4. Back-of-envelope math *(no GPU needed)*
5. Inspect and micro-benchmark your GPU
6. First generation with the offline `LLM` engine
7. One engine, three layers of configuration
8. Sampling parameters
9. Chat: `llm.chat` applies the template for you
10. Prompt formats: text, token IDs, chat, and pre-filling the answer
11. Batching: sequential vs batched throughput
12. Measured vs theoretical decode speed
13. Launch the server
14. Streaming & measuring TTFT / TPOT yourself
15. Concurrency sweep: the latency–throughput trade-off, live
16. Reading the `/metrics` endpoint

## 1. Setup & GPU check

Install vLLM plus a few helpers. On Colab, pick **Runtime → Change runtime type → L4 GPU** first.

> **Watch out.** vLLM pins a specific PyTorch/CUDA build. Install it into a **fresh** environment (venv/conda/Colab) to avoid version conflicts.

In [ ]:
import torch, subprocess, platform
print("Python :", platform.python_version())
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU    : {p.name} | {p.total_memory/1e9:.1f} GB | compute capability {p.major}.{p.minor}")
try:
    import vllm; print("vLLM   :", vllm.__version__)
except ImportError:
    print("vLLM not installed (fine for the *(no GPU needed)* cells)")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)

### Shared configuration & helpers

Run this cell **every time you restart the kernel**. Each Part only depends on it.

In [ ]:
import os, gc, time, json, statistics as st
import torch
import pandas as pd
import matplotlib.pyplot as plt

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # 1.5B params, 28 layers, GQA with 2 KV heads
MAX_LEN = 4096                          # context we allow per request (smaller = more room for KV cache)
plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": .3})

def shutdown(llm):
    # Release the GPU memory held by an offline vLLM engine.
    # If memory still isn't freed (it happens in notebooks), restart the kernel
    # and re-run this config cell; each Part is self-contained.
    try:
        llm.llm_engine.engine_core.shutdown()
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def gpu_mem():
    if not torch.cuda.is_available():
        return "no GPU"
    free, total = torch.cuda.mem_get_info()
    return f"GPU memory used: {(total-free)/1e9:.1f} / {total/1e9:.1f} GB"

print("Config ready:", MODEL)

## 2. Tokens: what the model actually sees *(no GPU needed)*

An LLM never sees text. A **tokenizer** turns text into integer IDs from a fixed vocabulary, and every cost, limit and latency number in inference is counted in **tokens**.

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL)

text = "Inference engineering is surprisingly fun!"
ids = tok.encode(text)
print(f"{len(text)} characters → {len(ids)} tokens   (vocab size: {tok.vocab_size:,})\n")
for i in ids:
    print(f"{i:>7}  →  {tok.decode([i])!r}")

**Try it:** tokenize code, a non-English sentence, or a long number. Notice how the characters-per-token ratio changes, and with it cost and latency.

In [ ]:
samples = {
    "English": "The quick brown fox jumps over the lazy dog.",
    "Code":    "def add(a, b):\n    return a + b",
    "Hindi":   "मशीन लर्निंग बहुत रोचक है",
    "Number":  "3.14159265358979323846",
}
for name, s in samples.items():
    n = len(tok.encode(s))
    print(f"{name:8} {len(s):3} chars → {n:3} tokens  ({len(s)/n:.1f} chars/token)")

### Chat templates

Chat models expect messages wrapped in special tokens. The **chat template** does this. It's the exact string that gets *prefilled*.

In [ ]:
messages = [{"role": "system", "content": "You are a concise assistant."},
            {"role": "user", "content": "What is a KV cache?"}]
prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)
print("Prompt tokens:", len(tok.encode(prompt)))

## 3. Generation from scratch: prefill, decode and the KV cache in 30 lines *(runs on CPU)*

Before any engine, here is the whole algorithm with a plain Hugging Face model. Read the two loops: the **prefill** is one forward pass over the entire prompt; **decode** is one forward pass per new token. With the KV cache, each decode step feeds in a single token and reuses the keys and values of everything before it. Without it, every step re-reads the whole sequence.

In [ ]:
import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer

SMALL = "Qwen/Qwen2.5-0.5B-Instruct"
dev = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if dev == "cuda" else torch.float32
tok_s = AutoTokenizer.from_pretrained(SMALL)
small = AutoModelForCausalLM.from_pretrained(SMALL, torch_dtype=dtype).to(dev).eval()

@torch.no_grad()
def generate(prompt, max_new=40, use_cache=True):
    seq = tok_s(prompt, return_tensors="pt").input_ids.to(dev)
    t0 = time.perf_counter()
    out = small(seq, use_cache=use_cache)                      # PREFILL: all prompt tokens in one pass
    nxt = out.logits[:, -1].argmax(-1, keepdim=True)           # greedy pick of the first new token
    past = out.past_key_values
    ttft = time.perf_counter() - t0
    seq = torch.cat([seq, nxt], 1)
    t1 = time.perf_counter()
    for _ in range(max_new - 1):                               # DECODE: one pass per token
        if use_cache:
            out = small(nxt, past_key_values=past, use_cache=True)   # feed 1 token, reuse cached K/V
            past = out.past_key_values
        else:
            out = small(seq, use_cache=False)                         # re-process the whole sequence
        nxt = out.logits[:, -1].argmax(-1, keepdim=True)
        seq = torch.cat([seq, nxt], 1)
    tpot = (time.perf_counter() - t1) / (max_new - 1)
    return tok_s.decode(seq[0], skip_special_tokens=True), ttft, tpot

prompt = tok_s.apply_chat_template([{"role": "user", "content": "In two sentences: why is LLM decoding limited by memory bandwidth?"}],
                                  tokenize=False, add_generation_prompt=True)
generate(prompt, max_new=4)                                    # warm-up, so the first timing isn't inflated
for cache in (True, False):
    text, ttft, tpot = generate(prompt, use_cache=cache)
    print(f"KV cache {'on ' if cache else 'off'}: TTFT {ttft*1e3:6.0f} ms   TPOT {tpot*1e3:6.1f} ms/token")
print("\n" + text.split("assistant")[-1].strip())

In [ ]:
# How big is that cache? Per token: 2 (K and V) x layers x KV heads x head_dim x bytes
c = small.config
hd = c.hidden_size // c.num_attention_heads
per_tok = 2 * c.num_hidden_layers * c.num_key_value_heads * hd * 2          # BF16 = 2 bytes
print(f"{SMALL}: {c.num_hidden_layers} layers, {c.num_key_value_heads} KV heads of dim {hd}")
print(f"KV cache: {per_tok/1024:.0f} KiB per token -> {per_tok*4096/2**20:.0f} MiB for one 4K-token conversation")
print("Every decode step reads all the weights plus this cache for every sequence in the batch.")

What to notice: with the cache, TPOT stays roughly flat as the sequence grows; without it, each step re-runs prefill on a longer sequence, so TPOT climbs. Engines like vLLM run exactly this loop, plus batching (many sequences per step), paging (where the cache lives) and fast kernels. Everything in Sessions 2 and 3 is an optimisation of these two loops.

## 4. Back-of-envelope math *(no GPU needed)*

Before touching a GPU, an inference engineer can predict most of what will happen with three formulas:

| Quantity | Formula |
|---|---|
| **Weights** | `params × bytes_per_param` |
| **KV cache per token** | `2 (K and V) × layers × kv_heads × head_dim × bytes` |
| **Decode speed ceiling** (batch 1) | `memory_bandwidth ÷ bytes_read_per_step` |

We read the real architecture from the model's config.

In [ ]:
from transformers import AutoConfig
cfg = AutoConfig.from_pretrained(MODEL)
head_dim = getattr(cfg, "head_dim", None) or cfg.hidden_size // cfg.num_attention_heads
print(f"layers={cfg.num_hidden_layers}  q_heads={cfg.num_attention_heads}  "
      f"kv_heads={cfg.num_key_value_heads}  head_dim={head_dim}")

def kv_bytes_per_token(layers, kv_heads, head_dim, bytes_per_elem=2):
    return 2 * layers * kv_heads * head_dim * bytes_per_elem

MODELS = {   # name: (params in billions, layers, kv_heads, head_dim)
    "Qwen2.5-1.5B":  (1.54, cfg.num_hidden_layers, cfg.num_key_value_heads, head_dim),
    "Llama-3.1-8B":  (8.0, 32, 8, 128),
    "Llama-3.1-70B": (70.6, 80, 8, 128),
}
print(f"\n{'model':15} {'weights BF16':>13} {'KV/token':>10} {'8K-token chat':>14}")
for name, (p, L, H, D) in MODELS.items():
    kv = kv_bytes_per_token(L, H, D)
    print(f"{name:15} {p*2:>10.1f} GB {kv/1024:>7.0f} KB {kv*8192/1e9:>11.2f} GB")

### How many users fit?

With an 80 GB GPU and a 0.9 memory budget, everything not used by weights becomes KV cache. That decides the maximum concurrency.

In [ ]:
def max_concurrent(params_b, L, H, D, gpu_gb=80, util=0.9, ctx=8192, w_bytes=2, kv_bytes=2):
    free = gpu_gb * util - params_b * w_bytes          # GB left for KV cache
    per_seq = kv_bytes_per_token(L, H, D, kv_bytes) * ctx / 1e9
    return max(0, int(free / per_seq))

fig, ax = plt.subplots()
ctxs = [1024, 2048, 4096, 8192, 16384, 32768]
for name, (p, L, H, D) in MODELS.items():
    if p * 2 < 72:   # 70B BF16 doesn't fit on one 80 GB GPU at all
        ax.plot(ctxs, [max_concurrent(p, L, H, D, ctx=c) for c in ctxs], marker="o", label=f"{name} BF16")
ax.plot(ctxs, [max_concurrent(70.6, 80, 8, 128, ctx=c, w_bytes=0.5) for c in ctxs], marker="s", ls="--",
        label="Llama-3.1-70B INT4")
ax.set(xscale="log", yscale="log", xlabel="context length per sequence (tokens)",
       ylabel="max concurrent sequences", title="Concurrency that fits on one 80 GB GPU")
ax.legend(); plt.show()

> Small models with few KV heads (Qwen2.5-1.5B has only **2**) can hold *thousands* of concurrent sequences. For big models, **context length × concurrency** is what runs out first, not weights.

### The decode "speed of light"

At batch size 1, generating each token means streaming *all* the weights from HBM to the compute units once.

In [ ]:
BANDWIDTH_GBs = {"H200": 4800, "H100": 3350, "A100": 2039, "L40S": 864, "A10G": 600, "L4": 300, "T4": 320}

def decode_ceiling(params_b, bytes_per_param, bw_gbs):
    return bw_gbs / (params_b * bytes_per_param)    # tokens/s, upper bound, batch 1

for gpu in ["H100", "A100", "L4"]:
    print(f"{gpu:5}: 8B BF16 ≤ {decode_ceiling(8, 2, BANDWIDTH_GBs[gpu]):5.0f} tok/s | "
          f"8B FP8 ≤ {decode_ceiling(8, 1, BANDWIDTH_GBs[gpu]):5.0f} tok/s | "
          f"1.5B BF16 ≤ {decode_ceiling(1.54, 2, BANDWIDTH_GBs[gpu]):5.0f} tok/s")

We'll compare these predictions to measured numbers in §12.

The slides quote spec-sheet numbers. Here we measure **your** GPU's two numbers that matter (memory bandwidth and matmul throughput) and derive its ops:byte ratio, the ridge point of the roofline.

| GPU | Memory | Bandwidth | BF16 dense TFLOPS | ops:byte | Notes |
|---|---|---|---|---|---|
| T4 | 16 GB GDDR6 | 0.32 TB/s | 65 (FP16) | ~200 | No BF16 → use `--dtype half`; no FlashAttention 2 |
| L4 | 24 GB GDDR6 | 0.30 TB/s | 121 | ~400 | FP8 capable (Ada) |
| A100 80GB | HBM2e | 2.0 TB/s | 312 | ~156 | No FP8 |
| H100 SXM | 80 GB HBM3 | 3.35 TB/s | 989 | ~295 | FP8; NVLink 900 GB/s |
| H200 | 141 GB HBM3e | 4.8 TB/s | 989 | ~206 | Same compute as H100, more/faster memory: a decode GPU |
| B200 | 192 GB HBM3e | ~8 TB/s | ~2,250 | ~280 | FP4 tensor cores; NVLink 1.8 TB/s |
| MI300X | 192 GB HBM3 | 5.3 TB/s | ~1,300 | ~245 | ROCm; vLLM & SGLang both support it |
| RTX 4090 | 24 GB GDDR6X | 1.0 TB/s | ~165 | ~165 | No NVLink → TP over PCIe is slow |
| RTX 5090 | 32 GB GDDR7 | 1.8 TB/s | ~210 | ~117 | FP4 (consumer Blackwell, sm_120) |
| Mac M3 Ultra | up to 512 GB unified | 0.82 TB/s | n/a | n/a | Huge memory, modest bandwidth: big MoE fits, decode is slowish |
| DGX Spark (GB10) | 128 GB unified | 0.27 TB/s | n/a | n/a | Fits 120B-class MoE; decode bandwidth-limited |

Spec-sheet numbers; vendor "sparse" TFLOPS are 2× these and rarely relevant to LLMs.

## 5. Inspect and micro-benchmark your GPU

In [ ]:
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    cc = f"{p.major}.{p.minor}"
    print(f"{p.name} | {p.total_memory/1e9:.0f} GB | {p.multi_processor_count} SMs | compute capability sm_{p.major}{p.minor}")
    print("BF16 native:", p.major >= 8, "| FP8 tensor cores:", (p.major, p.minor) >= (8, 9), "| FP4 tensor cores:", p.major >= 10)

    # 1) Memory bandwidth: copy a 2 GB buffer (read + write)
    x = torch.empty(2 * 1024**3 // 2, dtype=torch.bfloat16, device="cuda"); y = torch.empty_like(x)
    for _ in range(3): y.copy_(x)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    for _ in range(20): y.copy_(x)
    torch.cuda.synchronize(); bw = 2 * x.numel() * 2 * 20 / (time.perf_counter() - t0) / 1e12
    del x, y

    # 2) Matmul throughput: large BF16 GEMM (dtype half on pre-Ampere)
    dt = torch.bfloat16 if p.major >= 8 else torch.float16
    a = torch.randn(8192, 8192, dtype=dt, device="cuda"); b = torch.randn_like(a)
    for _ in range(3): a @ b
    torch.cuda.synchronize(); t0 = time.perf_counter()
    for _ in range(20): a @ b
    torch.cuda.synchronize(); tflops = 2 * 8192**3 * 20 / (time.perf_counter() - t0) / 1e12
    del a, b; torch.cuda.empty_cache()

    print(f"\nmeasured bandwidth ≈ {bw:.2f} TB/s | measured {dt} matmul ≈ {tflops:.0f} TFLOPS")
    print(f"→ your ridge point (ops:byte) ≈ {tflops/bw:.0f} FLOPs/byte")
    print(f"→ batch-1 decode ceiling for this 1.5B BF16 model ≈ {bw*1e12/(1.54e9*2):.0f} tok/s")

In [ ]:
# Interconnect: how GPUs talk to each other. NV# = NVLink, PIX/PHB/SYS = PCIe paths (slower → prefer PP or DP over TP)
!nvidia-smi topo -m 2>/dev/null || echo "single GPU / no nvidia-smi"

> Measured bandwidth usually lands at 80–90% of spec; matmul at 60–80% of spec. Those *achievable* numbers, not the spec sheet, are what your MBU and cost estimates should use.

### The memory hierarchy in one table (H100 SXM)

| Level | Size | Bandwidth | What lives there during inference |
|---|---|---|---|
| Registers + shared memory / L1 (per SM, 132 SMs) | ~256 KB per SM | far above HBM (on-chip) | FlashAttention tiles, GEMM tiles |
| L2 cache | 50 MB | several × HBM | Hot activations; too small for weights |
| HBM3 | 80 GB | 3.35 TB/s | **Weights + KV cache**: what decode streams every step |
| NVLink 4 (GPU↔GPU) | — | 900 GB/s | Tensor-parallel all-reduces, KV transfer |
| PCIe Gen5 x16 (GPU↔CPU) | — | ~64 GB/s | CPU offload, KV offload, loading weights |
| CPU DRAM | 100s of GB–TBs | ~0.1–0.5 TB/s | Offloaded experts/KV (llama.cpp `--n-cpu-moe`, LMCache) |
| NVMe SSD | TBs | ~7–14 GB/s | Weight files; cold KV tier; your cold-start time |

Each step down is roughly an order of magnitude slower. Every "offload" trick trades capacity for bandwidth on this ladder.

## 6. First generation with the offline `LLM` engine

`vllm.LLM` is the **offline / batch** API: no server, just Python. Perfect for evals, data generation and batch jobs.

While it loads, **read the logs**. They tell you a lot:

| Log line contains | What it means |
|---|---|
| `Model loading took … GiB` | Memory used by the weights |
| `GPU KV cache size: N tokens` | Total tokens of KV cache that fit, which is your capacity |
| `Maximum concurrency for 4096 tokens per request: X` | How many *full-length* requests fit at once |
| `Capturing CUDA graphs` | vLLM pre-records decode kernels to kill CPU launch overhead |

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(model=MODEL, max_model_len=MAX_LEN, gpu_memory_utilization=0.85)
print(gpu_mem())

In [ ]:
params = SamplingParams(temperature=0.7, max_tokens=128)
out = llm.generate(["Explain in two sentences why LLM decoding is memory-bandwidth bound."], params)[0]

print(out.outputs[0].text)
print(f"\nprompt tokens: {len(out.prompt_token_ids)} | generated tokens: {len(out.outputs[0].token_ids)} "
      f"| finish reason: {out.outputs[0].finish_reason}")

> `finish_reason` is `stop` (hit EOS / a stop string) or `length` (hit `max_tokens`). Silent truncation by `length` is one of the most common production bugs.

## 7. One engine, three layers of configuration

`LLM(...)` takes the same arguments as `vllm serve` flags (`max_model_len` ↔ `--max-model-len`). They all land in a single `VllmConfig`, which you can inspect. `llm.get_metrics()` returns the same Prometheus metrics the server exposes at `/metrics`, without a server.

In [ ]:
llm = LLM(model=MODEL, max_model_len=MAX_LEN, gpu_memory_utilization=0.6, seed=0)
cfg = llm.llm_engine.vllm_config
print("model          :", cfg.model_config.model, "| dtype", cfg.model_config.dtype, "| max_model_len", cfg.model_config.max_model_len)
print("cache          : block_size", cfg.cache_config.block_size, "| kv dtype", cfg.cache_config.cache_dtype,
      "| prefix caching", cfg.cache_config.enable_prefix_caching)
print("scheduler      : max_num_seqs", cfg.scheduler_config.max_num_seqs, "| max_num_batched_tokens", cfg.scheduler_config.max_num_batched_tokens)
print("parallel       : TP", cfg.parallel_config.tensor_parallel_size, "| PP", cfg.parallel_config.pipeline_parallel_size)
print("compilation    :", cfg.compilation_config.mode, "| cudagraph mode", cfg.compilation_config.cudagraph_mode)

In [ ]:
llm.generate(["warm-up"] * 8, SamplingParams(max_tokens=16))
for m in llm.get_metrics():
    if any(k in m.name for k in ("kv_cache_usage", "prefix_cache", "num_requests", "generation_tokens", "prompt_tokens")):
        print(f"{m.name:45} {getattr(m, 'value', '')}")

## 8. Sampling parameters

Each decode step produces a probability distribution over the vocabulary. **Sampling** picks one token from it.

| Parameter | Effect |
|---|---|
| `temperature` | `0` = greedy (deterministic). Higher = flatter distribution, more diverse |
| `top_p` / `top_k` / `min_p` | Cut the long tail of unlikely tokens |
| `n` | Several completions per prompt; they **share** the prompt's KV cache |
| `seed` | Reproducible sampling |
| `stop` | Stop strings |
| `logprobs` | Return the top-k probabilities at each step |

In [ ]:
prompt = "Give me a creative name for a GPU inference startup:"
for t in [0.0, 0.7, 1.5]:
    sp = SamplingParams(temperature=t, n=3, max_tokens=12, seed=42, stop=["\n"])
    outs = llm.generate([prompt], sp)[0].outputs
    print(f"temperature={t}:", [o.text.strip() for o in outs])

### Peeking at the probability distribution

`logprobs=5` returns the 5 most likely candidates at every step, so you can see exactly what the sampler chose from.

In [ ]:
import math
sp = SamplingParams(temperature=0, max_tokens=6, logprobs=5)
o = llm.generate(["The capital of France is"], sp)[0].outputs[0]
for step, (tid, cand) in enumerate(zip(o.token_ids, o.logprobs)):
    ranked = sorted(cand.items(), key=lambda kv: -kv[1].logprob)
    shown = "  ".join(f"{lp.decoded_token!r}:{math.exp(lp.logprob):.2f}" for _, lp in ranked)
    print(f"step {step}: chose {tok.decode([tid])!r:12} | {shown}")

## 9. Chat: `llm.chat` applies the template for you

In [ ]:
conversation = [
    {"role": "system", "content": "You are an expert inference engineer. Answer in 3 bullet points."},
    {"role": "user", "content": "How do I reduce time-to-first-token?"},
]
print(llm.chat(conversation, SamplingParams(temperature=0.3, max_tokens=200))[0].outputs[0].text)

## 10. Prompt formats: text, token IDs, chat, and pre-filling the answer

`generate` accepts plain strings, `{"prompt": ...}` dicts, or `{"prompt_token_ids": [...]}` when you've tokenized yourself (handy when the same tokens feed a training pipeline). `chat` applies the model's chat template and accepts a few options worth knowing:

| `llm.chat(...)` argument | What it does |
|---|---|
| `continue_final_message=True, add_generation_prompt=False` | Pre-fill the start of the assistant's answer and let the model continue it. The cheapest way to force a format. |
| `chat_template_kwargs={...}` | Passed into the Jinja template, e.g. `{"enable_thinking": False}` for Qwen3 |
| `tools=[...]` | Renders tool schemas into the prompt (the server parses the calls back out, see Session 3 §12) |
| `chat_template=...` | Override a broken or missing template |

In [ ]:
sp = SamplingParams(temperature=0, max_tokens=40)
tok = llm.get_tokenizer()
ids = tok.encode("The three things that decide decode speed are")
print("token IDs in :", llm.generate([{"prompt_token_ids": ids}], sp)[0].outputs[0].text.strip()[:120])

msgs = [{"role": "user", "content": "List three GPUs good for LLM inference."},
        {"role": "assistant", "content": "1. NVIDIA H100\n2."}]           # pre-filled answer
out = llm.chat(msgs, sp, add_generation_prompt=False, continue_final_message=True)
print("continued    : 1. NVIDIA H100\n2." + out[0].outputs[0].text[:120])

## 11. Batching: sequential vs batched throughput

Decode is memory-bound: each step reads every weight once *whether it serves 1 sequence or 100*. So batching many sequences into the same step is nearly free, up to the point where compute becomes the bottleneck.

We generate the same 64 prompts two ways:
1. **One at a time** (what a naive server does)
2. **All at once** (vLLM's continuous batching)

In [ ]:
topics = ["GPUs", "KV caches", "tokenizers", "quantization", "batching", "attention", "transformers", "latency"]
prompts = [f"Write a short paragraph about {topics[i % len(topics)]} (variant {i})." for i in range(64)]
sp = SamplingParams(temperature=0.8, max_tokens=128, ignore_eos=True)   # fixed length → fair comparison

t0 = time.perf_counter()
for p in prompts[:8]:                      # only 8 sequentially, it's slow
    llm.generate([p], sp, use_tqdm=False)
seq_tps = 8 * 128 / (time.perf_counter() - t0)

t0 = time.perf_counter()
llm.generate(prompts, sp, use_tqdm=False)
batch_tps = 64 * 128 / (time.perf_counter() - t0)

print(f"sequential: {seq_tps:7.0f} output tokens/s")
print(f"batched   : {batch_tps:7.0f} output tokens/s   →  {batch_tps/seq_tps:.0f}× more throughput, same GPU")

### How far does it scale?

Sweep the batch size and watch throughput climb, then flatten as we move from memory-bound toward compute-bound (the roofline from the slides).

In [ ]:
results = []
for bs in [1, 2, 4, 8, 16, 32, 64, 128, 256]:
    batch = (prompts * 4)[:bs]
    t0 = time.perf_counter()
    llm.generate(batch, sp, use_tqdm=False)
    dt = time.perf_counter() - t0
    results.append((bs, bs * 128 / dt, 128 / dt))
    print(f"batch {bs:4}: {bs*128/dt:8.0f} tok/s total | {128/dt:6.1f} tok/s per sequence")

bs_, tot, per = zip(*results)
fig, ax1 = plt.subplots()
ax1.plot(bs_, tot, "o-", color="tab:purple", label="total throughput"); ax1.set_xscale("log", base=2)
ax1.set(xlabel="batch size", ylabel="total tokens/s")
ax2 = ax1.twinx(); ax2.plot(bs_, per, "s--", color="tab:red", label="per-sequence speed"); ax2.set_ylabel("tokens/s per sequence")
fig.legend(loc="upper center", ncol=2); plt.title("Batching: throughput vs per-user speed"); plt.show()

> Total throughput goes up roughly linearly at first while per-sequence speed barely drops. That's the memory-bound regime, where batching is free. Eventually per-sequence speed falls: the GPU is now compute-bound. **Every production deployment picks a point on this curve.**

## 12. Measured vs theoretical decode speed

Compare the batch-1 decode speed you just measured with the bandwidth ceiling from §4.

In [ ]:
gpu_name = torch.cuda.get_device_name(0)
bw = next((v for k, v in BANDWIDTH_GBs.items() if k in gpu_name), None)
measured = results[0][2]   # batch-1 tokens/s from the sweep (includes a little prefill time)
if bw:
    ceiling = decode_ceiling(1.54, 2, bw)
    print(f"{gpu_name}: ceiling ≈ {ceiling:.0f} tok/s | measured ≈ {measured:.0f} tok/s "
          f"→ {measured/ceiling:.0%} of speed-of-light")
else:
    print(f"Unknown bandwidth for {gpu_name}; add it to BANDWIDTH_GBs. Measured: {measured:.0f} tok/s")

> For a model this small, fixed per-step overheads (kernel launches, sampling, Python) are a big share, so you'll often land well below the ceiling. For 8B+ models the measured number gets much closer to `bandwidth ÷ bytes`.

**Free the GPU before Part B:**

In [ ]:
shutdown(llm); del llm
print(gpu_mem())

In production you run vLLM as an **OpenAI-compatible HTTP server**. Any OpenAI SDK, LangChain, or `curl` works against it unchanged.

```
client ──HTTP──▶ API server (tokenize, chat template) ──▶ EngineCore (scheduler + KV cache manager) ──▶ GPU worker
       ◀──SSE stream── detokenize ◀───────────────────────────────── sampled tokens ◀────────────────────┘
```

## 13. Launch the server

We start it in the background and write logs to `vllm_server.log`. In a terminal you'd simply run:

```bash
vllm serve Qwen/Qwen2.5-1.5B-Instruct --max-model-len 4096 --gpu-memory-utilization 0.85
```

In [ ]:
import subprocess, urllib.request

PORT = 8000
BASE = f"http://localhost:{PORT}"
log = open("vllm_server.log", "w")
server = subprocess.Popen(
    ["vllm", "serve", MODEL, "--port", str(PORT),
     "--max-model-len", str(MAX_LEN), "--gpu-memory-utilization", "0.85"],
    stdout=log, stderr=subprocess.STDOUT)

def wait_healthy(timeout=600):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if server.poll() is not None:
            raise RuntimeError("server exited; see vllm_server.log")
        try:
            if urllib.request.urlopen(f"{BASE}/health", timeout=2).status == 200:
                return time.time() - t0
        except Exception:
            time.sleep(2)
    raise TimeoutError("server not healthy; see vllm_server.log")

print(f"server ready in {wait_healthy():.0f}s (that's your cold start: weights load + compile + CUDA graphs)")

In [ ]:
# The capacity lines from the startup log
for line in open("vllm_server.log"):
    if any(k in line for k in ["KV cache size", "Maximum concurrency", "Model loading took"]):
        print(line.strip()[-140:])

## 14. Streaming & measuring TTFT / TPOT yourself

| Metric | Definition |
|---|---|
| **TTFT** | time from request sent → first content token received |
| **TPOT / ITL** | average gap between subsequent tokens = `(t_last − t_first) / (N − 1)` |
| **E2E** | `TTFT + (N−1) × TPOT` |

In [ ]:
from openai import OpenAI, AsyncOpenAI
client = OpenAI(base_url=f"{BASE}/v1", api_key="not-needed")

def measure_stream(messages, max_tokens=200, **kw):
    t0 = time.perf_counter(); first = last = None; n = 0; text = []
    stream = client.chat.completions.create(model=MODEL, messages=messages, max_tokens=max_tokens,
                                            stream=True, stream_options={"include_usage": True}, **kw)
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            now = time.perf_counter(); first = first or now; last = now
            text.append(chunk.choices[0].delta.content)
        if chunk.usage:
            n = chunk.usage.completion_tokens
    ttft = first - t0
    tpot = (last - first) / max(n - 1, 1)
    return {"ttft_ms": ttft * 1e3, "tpot_ms": tpot * 1e3, "tok_per_s": 1 / tpot, "tokens": n,
            "e2e_s": last - t0, "text": "".join(text)}

r = measure_stream([{"role": "user", "content": "Explain continuous batching to a new engineer."}])
print(r["text"][:400], "...\n")
print({k: round(v, 1) for k, v in r.items() if k != "text"})

## 15. Concurrency sweep: the latency–throughput trade-off, live

We fire N simultaneous requests and record per-request TTFT and TPOT plus total system throughput. This is the same curve from the slides, measured on your GPU.

In [ ]:
import asyncio
aclient = AsyncOpenAI(base_url=f"{BASE}/v1", api_key="not-needed")

async def one_request(i, max_tokens=256):
    t0 = time.perf_counter(); first = last = None; n = 0
    stream = await aclient.chat.completions.create(
        model=MODEL, max_tokens=max_tokens, stream=True, stream_options={"include_usage": True},
        extra_body={"ignore_eos": True},   # fixed output length → comparable numbers
        messages=[{"role": "user", "content": f"Write a story about GPU number {i}."}])
    async for c in stream:
        if c.choices and c.choices[0].delta.content:
            now = time.perf_counter(); first = first or now; last = now
        if c.usage: n = c.usage.completion_tokens
    return first - t0, (last - first) / max(n - 1, 1), n

async def sweep(levels=(1, 4, 16, 64, 128)):
    rows = []
    for c in levels:
        t0 = time.perf_counter()
        res = await asyncio.gather(*[one_request(i) for i in range(c)])
        wall = time.perf_counter() - t0
        ttfts, tpots, ns = zip(*res)
        rows.append({"concurrency": c, "ttft_p50_ms": st.median(ttfts) * 1e3,
                     "ttft_max_ms": max(ttfts) * 1e3, "tpot_p50_ms": st.median(tpots) * 1e3,
                     "per_user_tok_s": 1 / st.median(tpots), "system_tok_s": sum(ns) / wall})
        print(rows[-1])
    return rows

rows = await sweep()

In [ ]:
import pandas as pd
df = pd.DataFrame(rows).round(1); display(df)

fig, ax = plt.subplots()
ax.plot(df.system_tok_s, df.per_user_tok_s, "o-")
for _, r in df.iterrows():
    ax.annotate(f"c={int(r.concurrency)}", (r.system_tok_s, r.per_user_tok_s), textcoords="offset points", xytext=(6, 4))
ax.set(xlabel="system throughput (tokens/s)", ylabel="per-user speed (tokens/s)",
       title="Every deployment picks a point on this curve")
plt.show()

> Read the table like an SLO owner: *"We need ≥ 30 tok/s per user and TTFT P50 < 500 ms. What's the highest concurrency that still meets that?"* The system throughput **at that point** is your **goodput**, and it sets how many GPUs you need.

## 16. Reading the `/metrics` endpoint

vLLM exposes Prometheus metrics. These are what you'd put on a Grafana dashboard and autoscale on.

In [ ]:
import re
raw = urllib.request.urlopen(f"{BASE}/metrics").read().decode()
KEYS = ["num_requests_running", "num_requests_waiting", "kv_cache_usage", "cache_usage",
        "prefix_cache_queries", "prefix_cache_hits", "prompt_tokens_total", "generation_tokens_total",
        "time_to_first_token_seconds_sum", "time_to_first_token_seconds_count", "num_preemptions"]
for line in raw.splitlines():
    if line.startswith("vllm:") and any(k in line for k in KEYS):
        name = re.sub(r"\{.*\}", "", line.split()[0]); print(f"{name:55} {float(line.split()[-1]):,.3f}")

| Metric | Why you watch it |
|---|---|
| `num_requests_waiting` | Queueing: **the** autoscaling signal |
| `kv_cache_usage_perc` | Near 100% → preemptions and latency spikes |
| `prefix_cache_hits / queries` | Hit rate; low means wasted prefill |
| `time_to_first_token_seconds` | TTFT histogram → P50/P99 |
| `num_preemptions` | Requests evicted for lack of KV memory |

Metric names can change slightly between vLLM versions; the filter above matches by substring.

## Exercises

1. **Predict, then measure.** Before running §12 on a new GPU, write down the decode ceiling from §4's formula. How close does the measured number get, and where does the gap go?
2. **Find your goodput.** In §15, find the highest concurrency where every user still gets 30 tokens/s or more. What's the system throughput there?
3. **Size a real deployment.** With §4's functions: how many concurrent 8K-token chats fit for Llama-3.1-8B on one H100, in BF16 and with an FP8 KV cache?
4. **Break the cache.** In §3, run the generation loop with `use_cache=False` for 200 tokens and plot TPOT per step. Why does it grow?

In [ ]:
# Stop anything this notebook started
for _name in ("server", "sg", "trt"):
    if isinstance(globals().get(_name), subprocess.Popen):
        stop(globals()[_name])
print(gpu_mem())

Next: **Session 2, Intermediate**: continuous batching, paging, quantization and speculative decoding, each built from scratch and then measured in vLLM. [`Session2_Intermediate.ipynb`](Session2_Intermediate.ipynb)